# t-SNE: t-Distributed Stochastic Neighbor Embedding

**Topic:** Unsupervised Learning — Dimensionality Reduction

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, FloatSlider, Output, HBox, VBox
from IPython.display import display, clear_output, Markdown
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from tkh_utils import PALETTE, FONT, base_layout, load_california_housing


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** how t-SNE preserves local neighborhood structure in a low-dimensional projection
- **Explain** the role of perplexity and why it controls the balance between local and global structure
- **Interpret** a t-SNE plot correctly — including what you can and cannot conclude from cluster distances

> **Tip:** Run the perplexity widget with values from 5 to 50. Notice how low perplexity creates many small tight clusters while high perplexity creates smoother, more global structure. Neither extreme is wrong — they show different levels of detail.

---
## How we got here

In **[07_pca.ipynb](07_pca.ipynb)** you learned that PCA finds linear projections that preserve variance. Many real-world datasets have non-linear structure — the clusters in the data don't separate along straight lines.

In **[math/linear_algebra/06_norms_and_distances.ipynb](../math/linear_algebra/06_norms_and_distances.ipynb)** you learned how distances are computed. t-SNE starts from pairwise distances in the original space and finds a 2D layout that preserves the neighborhood structure — not the variance.

---
## Why this matters for data science

PCA is fast and interpretable but can miss non-linear structure. t-SNE is the standard visualization tool when PCA produces an uninformative scatter. It is widely used in:
- Single-cell RNA sequencing (revealing cell types)
- Natural language processing (visualizing word embeddings)
- Computer vision (visualizing image feature spaces)

The key limitation: t-SNE is for visualization only. The cluster sizes and inter-cluster distances in a t-SNE plot do not have reliable geometric meaning. Use it to discover structure — not to measure it.

In **[ml_concepts/11_the_curse_of_dimensionality.ipynb](../ml_concepts/11_the_curse_of_dimensionality.ipynb)** you learned why distance-based methods degrade in high dimensions. t-SNE is one direct response: it recomputes a whole new notion of "neighbor" that works in the low-dimensional layout, rather than trusting raw high-dimensional distances.

---
## Where it sits on the spectrum

Referencing **[ml_concepts/13_interpretability_vs_complexity.ipynb](../ml_concepts/13_interpretability_vs_complexity.ipynb)**:

**Interpretability: Low.** t-SNE axes have no unit or meaning. Cluster distances are not proportional to true distances. The algorithm is also stochastic — two runs with different seeds produce different plots.

**Complexity: High.** Exact t-SNE is O(n²) in both time and memory. Barnes-Hut approximation (sklearn default) reduces this to O(n log n), making it feasible for up to ~50,000 samples.

**Position:** Low interpretability, high complexity — a specialized visualization tool, not a general-purpose feature engineering method.

---
## How it learns

t-SNE converts high-dimensional data into a 2D (or 3D) layout that preserves local neighborhood relationships.

**Step 1 — High-dimensional similarities.** For each pair of points, compute a Gaussian probability that expresses how likely they are to be neighbors: nearby points get high probability; distant points get near-zero probability. The "perplexity" parameter controls how many effective neighbors each point considers.

**Step 2 — Low-dimensional layout.** Place all points randomly in 2D. Compute a similar neighborhood probability in 2D using a Student's t-distribution (heavier tails than Gaussian).

**Step 3 — Minimize divergence.** Iteratively move points in 2D to minimize the KL divergence between the high-dimensional and low-dimensional neighborhood distributions. Similar high-dimensional neighbors get pulled together; dissimilar points get pushed apart.

**Why t-distribution?** The t-distribution's heavy tails allow moderately similar points to be placed farther apart in 2D — preventing the "crowding problem" where many dissimilar points get squeezed together in the center.

---
## The math behind it

**High-dimensional similarity** (symmetric):
$$p_{ij} = \frac{p_{j|i} + p_{i|j}}{2n}, \quad p_{j|i} = \frac{\exp\left(-\|x_i - x_j\|^2 / 2\sigma_i^2\right)}{\sum_{k \neq i} \exp\left(-\|x_i - x_k\|^2 / 2\sigma_i^2\right)}$$
Where $\sigma_i$ is chosen so that $p_{j|i}$ has effective number of neighbors equal to perplexity = $2^{H(P_i)}$.

**Low-dimensional similarity** (Student t with 1 degree of freedom):
$$q_{ij} = \frac{\left(1 + \|y_i - y_j\|^2\right)^{-1}}{\sum_{k \neq l}\left(1 + \|y_k - y_l\|^2\right)^{-1}}$$

**Cost function** (KL divergence):
$$C = \sum_{i \neq j} p_{ij} \log \frac{p_{ij}}{q_{ij}}$$

Minimizing $C$ with gradient descent moves 2D points so that $q_{ij}$ matches $p_{ij}$: points that are close in high dimensions end up close in 2D.

---
## Try it yourself

In [ ]:
out1 = Output()

perplexity_slider = IntSlider(
    value=30, min=5, max=50, step=5,
    description="Perplexity:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)

X, y = load_california_housing()
X_sc = StandardScaler().fit_transform(X)
price_tier = pd.qcut(y, q=3, labels=["Low", "Mid", "High"])

rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X_sc), size=1500, replace=False)
X_sample = X_sc[sample_idx]
tier_sample = price_tier.iloc[sample_idx].reset_index(drop=True)

tier_colors = {"Low": PALETTE["primary"], "Mid": PALETTE["accent"], "High": PALETTE["secondary"]}

def render_perplexity(perplexity):
    Z = TSNE(n_components=2, perplexity=perplexity, random_state=42, init="pca").fit_transform(X_sample)
    traces = []
    for tier, color in tier_colors.items():
        mask = (tier_sample == tier).to_numpy()
        traces.append(go.Scatter(
            x=Z[mask, 0], y=Z[mask, 1], mode="markers",
            marker=dict(color=color, size=5, opacity=0.6),
            name=f"{tier} price",
        ))
    layout = base_layout(
        title=f"t-SNE Projection — Perplexity = {perplexity} (≈{perplexity} effective neighbors per point)",
        xaxis_title="t-SNE dim 1",
        yaxis_title="t-SNE dim 2",
    )
    fig = go.Figure(data=traces, layout=layout)

    if perplexity <= 15:
        regime = "Low perplexity — expect many tight sub-clusters as each point sticks close to its nearest neighbors."
    elif perplexity >= 40:
        regime = "High perplexity — expect fewer, broader regions as more distant points get pulled into the same neighborhood."
    else:
        regime = "Close to the recommended default (≈30), balancing local detail against global smoothness."

    caption = f"**Perplexity = {perplexity} → ≈{perplexity} effective neighbors per point.** {regime}"

    with out1:
        clear_output(wait=True)
        fig.show()
        display(Markdown(caption))

def on_change_perplexity(change):
    render_perplexity(perplexity_slider.value)

perplexity_slider.observe(on_change_perplexity, names="value")
display(VBox([perplexity_slider, out1]))
render_perplexity(perplexity_slider.value)

In [ ]:
out2 = Output()

color_dropdown = Dropdown(
    options=[("Price tier", "tier"), ("Median income", "income"), ("Latitude", "lat")],
    value="tier",
    description="Color by:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)

Z_pca = PCA(n_components=2, random_state=42).fit_transform(X_sample)
Z_tsne30 = TSNE(n_components=2, perplexity=30, random_state=42, init="pca").fit_transform(X_sample)

income_sample = X.iloc[sample_idx]["MedInc"].to_numpy()
lat_sample = X.iloc[sample_idx]["Latitude"].to_numpy()

def make_traces(Z, values_key, show_legend_or_bar):
    if values_key == "tier":
        return [
            go.Scatter(
                x=Z[(tier_sample == t).to_numpy(), 0], y=Z[(tier_sample == t).to_numpy(), 1],
                mode="markers", marker=dict(color=c, size=5, opacity=0.6),
                name=f"{t} price", showlegend=show_legend_or_bar,
            )
            for t, c in tier_colors.items()
        ]
    vals = income_sample if values_key == "income" else lat_sample
    label = "Median income" if values_key == "income" else "Latitude"
    marker = dict(color=vals, colorscale="Viridis", size=5, opacity=0.7, showscale=show_legend_or_bar)
    if show_legend_or_bar:
        marker["colorbar"] = dict(title=label)
    return [go.Scatter(x=Z[:, 0], y=Z[:, 1], mode="markers", marker=marker, name=label, showlegend=False)]

CAPTIONS = {
    "tier": "**Coloring by price tier.** Just like in 07_pca, the tiers overlap heavily in both panels — geography and dwelling size dominate the structure here, not price.",
    "income": "**Coloring by median income.** Income correlates with this structure more directly than price tier did — look for a smoother gradient across the t-SNE clusters.",
    "lat": "**Coloring by latitude.** The gradient you see largely reflects Northern vs. Southern California — geography is one of the strongest signals in this dataset.",
}

def render_comparison(values_key):
    fig = make_subplots(rows=1, cols=2, subplot_titles=("PCA 2D Projection", "t-SNE 2D Projection (perplexity=30)"))
    for trace in make_traces(Z_pca, values_key, show_legend_or_bar=True):
        fig.add_trace(trace, row=1, col=1)
    for trace in make_traces(Z_tsne30, values_key, show_legend_or_bar=False):
        fig.add_trace(trace, row=1, col=2)
    fig.update_xaxes(title_text="PC1", row=1, col=1)
    fig.update_yaxes(title_text="PC2", row=1, col=1)
    fig.update_xaxes(title_text="t-SNE dim 1", row=1, col=2)
    fig.update_yaxes(title_text="t-SNE dim 2", row=1, col=2)
    fig.update_layout(
        base_layout(title="PCA vs. t-SNE", xaxis_title="", yaxis_title=""),
        height=450, showlegend=(values_key == "tier"),
    )
    with out2:
        clear_output(wait=True)
        fig.show()
        display(Markdown(CAPTIONS[values_key]))

def on_change_comparison(change):
    render_comparison(color_dropdown.value)

color_dropdown.observe(on_change_comparison, names="value")
display(VBox([color_dropdown, out2]))
render_comparison(color_dropdown.value)

---
## What's happening?

The perplexity widget shows that t-SNE has no single "correct" output — perplexity changes the level of detail. Perplexity ≈ 30 is the recommended default (Van der Maaten & Hinton, 2008), but datasets with very different sizes or densities may need different values. Watch the price tiers (Low/Mid/High) fragment into many tight sub-clusters at low perplexity and merge into smoother, more overlapping regions at high perplexity.

The comparison widget lets you re-color the same PCA and t-SNE projections by price tier, median income, or latitude. Notice that t-SNE often separates structure that PCA compresses into an overlapping blob — and switching the coloring variable reveals which underlying factor is actually driving the separation you're looking at.

**Critical t-SNE interpretation rules:**
1. Cluster membership is meaningful (proximity = similarity). ✓
2. Cluster sizes are NOT meaningful (large ≠ more spread in original space). ✗
3. Inter-cluster distances are NOT meaningful (far ≠ truly different). ✗
4. Two runs with different seeds give different plots — both may be valid. ⚠

---
## Key hyperparameters

**`perplexity`** (default 30): Effective number of neighbors considered per point. Try values between 5 and 50. Larger datasets generally need higher perplexity.

**`max_iter`** (default 1000, renamed from `n_iter` in scikit-learn 1.5 — you may still see `n_iter` in older tutorials): Number of optimization steps. Increase to 2000-5000 for better convergence on large datasets.

**`learning_rate`** (default `'auto'`): Step size for gradient descent. Auto-setting works for most cases — it computes the rate from your sample size (roughly, larger samples get a higher learning rate, with a floor of 50).

sklearn docs: [sklearn.manifold.TSNE](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html)

---
## Strengths and weaknesses

| Strength | Weakness |
|----------|----------|
| Reveals non-linear cluster structure PCA misses | Stochastic — different seeds give different plots |
| Standard tool for high-dimensional visualization | Slow: O(n log n) even with Barnes-Hut |
| Excellent at separating distinct clusters visually | Cluster distances not interpretable |
| Handles complex manifold structure | Cannot project new points without rerunning |
| Widely understood and published | Sensitive to perplexity choice |

---
## When to use it / When NOT to use it

| Use t-SNE when... | Do NOT use t-SNE when... |
|-------------------|--------------------------|
| PCA gives an uninformative scatter | You need to interpret the axes |
| Dataset has non-linear cluster structure | You need to project new data points |
| You want to visualize high-dimensional embeddings | Dataset is very large (>100,000 rows) |
| Cluster membership is what matters | You need reproducible, deterministic output |
| Exploring single-cell, NLP, or vision feature spaces | You need to measure inter-cluster distances |

---
## Key takeaway

> **t-SNE is a visualization tool, not a feature engineering tool — cluster membership in the plot is meaningful, but cluster sizes, shapes, and distances between clusters are not, and two runs on the same data produce different layouts.**

---
*Next up: 09_umap — faster than t-SNE and better at preserving global structure, at the cost of a more complex algorithm*